# Test: normalization modes in MANGODisplay_OSN

Runs `MakeSummaryMovies` for each of the three normalization modes and compares
sample frames and frame-to-frame pixel variance to confirm flicker reduction.

| Mode | What it does |
|------|--------------|
| `global` | One fixed percentile range across all sites and all frames (default) |
| `rolling` | Per-frame per-site limits smoothed with a running median — recommended anti-flicker |
| `per_frame` | Range recomputed per site per frame — maximum suppression, no cross-frame amplitude comparison |

**Edit the cells marked `# --- USER CONFIG ---` before running.**

## 1. Imports

In [ ]:
from datetime import datetime, timedelta
import os
from glob import glob

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np

from airglow.MANGODisplay_OSN import MakeSummaryMovies
from airglow.cloud_storage import CloudStorage, Configuration

## 2. User configuration

In [ ]:
# --- USER CONFIG ---
ENV_FILE = '.env'          # path to credentials file

YEAR = 2024
DOY  = 100                 # day-of-year to process

EMISSION = 'green'         # 'green' or 'red'

ROLLING_WINDOW = 11        # frames in the running-median window (forced odd)

# Separate output directories so runs do not overwrite each other
BASE_OUTPUT = '/tmp/mango_norm_test'
# -------------------

OUTPUT = {
    'global':    os.path.join(BASE_OUTPUT, 'global'),
    'rolling':   os.path.join(BASE_OUTPUT, 'rolling'),
    'per_frame': os.path.join(BASE_OUTPUT, 'per_frame'),
}
for d in OUTPUT.values():
    os.makedirs(d, exist_ok=True)

print(f"Date: {datetime(YEAR, 1, 1) + timedelta(days=DOY - 1):%Y-%m-%d}")
print(f"Emission: {EMISSION}")
print(f"Rolling window: {ROLLING_WINDOW} frames")
for k, v in OUTPUT.items():
    print(f"  {k:10s} -> {v}")

## 3. Cloud credentials and shared parameters

In [ ]:
config        = Configuration(ENV_FILE)
cloud_storage = CloudStorage(config)

date = datetime(YEAR, 1, 1) + timedelta(days=DOY - 1)

if EMISSION == 'green':
    sites_asi = ['bdr', 'blo', 'cfs', 'cvo', 'low', 'mro', 'new', 'ovi', 'pfo']
    sites_fpi = ['cvo', 'low', 'blo']
    Tlo, Thi  = 2, 20
else:
    sites_asi = ['cfs', 'cvo', 'eio', 'mdk', 'mto', 'par', 'lao']
    sites_fpi = ['cvo', 'low', 'blo', 'uao']
    Tlo, Thi  = 8, 30

base_analysis = {
    'date':           date,
    'emission':       EMISSION,
    'ntaps':          13,
    'download_data':  True,
    'el_cutoff':      20.,
    'sites_asi':      sites_asi,
    'sites_fpi':      sites_fpi,
    'Tlo':            Tlo,
    'Thi':            Thi,
    'rolling_window': ROLLING_WINDOW,
}

## 4. Run 1 — global normalization (baseline)

In [ ]:
system_parameters = {
    'ASI_directory': os.path.join(BASE_OUTPUT, 'asi'),
    'FPI_directory': os.path.join(BASE_OUTPUT, 'fpi'),
    'output_directory': OUTPUT['global'],
}

MakeSummaryMovies(
    system_parameters,
    {**base_analysis, 'normalization': 'global'},
    cloud_storage, config,
    delete_working_files=False,
    clim_percentile=(2, 98),
)

pngs = {k: sorted(glob(os.path.join(v, '*.png'))) for k, v in OUTPUT.items()}
print(f"global: {len(pngs['global'])} frames")

## 5. Run 2 — rolling normalization (recommended anti-flicker)

In [ ]:
# Data already downloaded in Run 1 — skip re-downloading.
system_parameters['output_directory'] = OUTPUT['rolling']

MakeSummaryMovies(
    system_parameters,
    {**base_analysis, 'normalization': 'rolling', 'download_data': False},
    cloud_storage, config,
    delete_working_files=False,
    clim_percentile=(2, 98),
)

pngs['rolling'] = sorted(glob(os.path.join(OUTPUT['rolling'], '*.png')))
print(f"rolling: {len(pngs['rolling'])} frames")

## 6. Run 3 — per-frame normalization (maximum suppression, for reference)

In [ ]:
system_parameters['output_directory'] = OUTPUT['per_frame']

MakeSummaryMovies(
    system_parameters,
    {**base_analysis, 'normalization': 'per_frame', 'download_data': False},
    cloud_storage, config,
    delete_working_files=False,
    clim_percentile=(2, 98),
)

pngs['per_frame'] = sorted(glob(os.path.join(OUTPUT['per_frame'], '*.png')))
print(f"per_frame: {len(pngs['per_frame'])} frames")

## 7. Side-by-side frame comparison

Each column is one time step; rows are the three normalization modes.
Pick frames near a cloudy or transient period to see the difference most clearly.

In [ ]:
# --- USER CONFIG ---
N_COMPARE = 4   # number of frames to show side-by-side
# -------------------

modes = ['global', 'rolling', 'per_frame']
n_frames = min(len(pngs[m]) for m in modes if pngs[m])

if n_frames == 0:
    print("No frames found — check the output directories.")
else:
    indices = np.linspace(0, n_frames - 1, N_COMPARE, dtype=int)

    fig, axes = plt.subplots(
        len(modes), N_COMPARE,
        figsize=(5 * N_COMPARE, 4 * len(modes)),
        constrained_layout=True,
    )
    fig.suptitle('Normalization mode comparison', fontsize=12)

    for row, mode in enumerate(modes):
        for col, idx in enumerate(indices):
            ax = axes[row, col]
            if idx < len(pngs[mode]):
                ax.imshow(mpimg.imread(pngs[mode][idx]))
            ax.set_title(f'{mode}  [frame {idx}]', fontsize=7)
            ax.axis('off')

    plt.show()

## 8. Frame-to-frame pixel variance

Lower std-of-std confirms more stable image contrast (less flicker) across the movie.

In [ ]:
def frame_std_series(png_list):
    """Return per-frame pixel std (float32 PNG values in [0, 1])."""
    return np.array([float(mpimg.imread(p).std()) for p in png_list])

colors = {'global': 'steelblue', 'rolling': 'seagreen', 'per_frame': 'darkorange'}

if n_frames > 0:
    stds = {m: frame_std_series(pngs[m]) for m in modes if pngs[m]}

    fig, ax = plt.subplots(figsize=(13, 4))
    for mode, s in stds.items():
        ax.plot(s, label=mode, color=colors[mode])
    ax.set_xlabel('Frame index')
    ax.set_ylabel('Image pixel std')
    ax.set_title('Frame-to-frame pixel contrast by normalization mode')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"{'Mode':<12} {'mean std':>10} {'std-of-std':>12}")
    print('-' * 36)
    for mode, s in stds.items():
        print(f"{mode:<12} {s.mean():>10.4f} {s.std():>12.4f}")

## 9. Tune rolling window size

Re-run just the rolling mode with different window sizes to find the right
balance between smoothness and responsiveness to real nightly intensity changes.

In [ ]:
# --- USER CONFIG ---
WINDOWS_TO_TEST = [5, 11, 21]   # frames (odd values recommended)
# -------------------

window_stds = {}
for win in WINDOWS_TO_TEST:
    out_dir = os.path.join(BASE_OUTPUT, f'rolling_{win}')
    os.makedirs(out_dir, exist_ok=True)
    system_parameters['output_directory'] = out_dir

    MakeSummaryMovies(
        system_parameters,
        {**base_analysis, 'normalization': 'rolling', 'rolling_window': win, 'download_data': False},
        cloud_storage, config,
        delete_working_files=False,
        clim_percentile=(2, 98),
    )

    these_pngs = sorted(glob(os.path.join(out_dir, '*.png')))
    if these_pngs:
        window_stds[win] = frame_std_series(these_pngs)
    print(f"window={win}: {len(these_pngs)} frames")

if window_stds:
    fig, ax = plt.subplots(figsize=(13, 4))
    for win, s in window_stds.items():
        ax.plot(s, label=f'rolling w={win}')
    ax.set_xlabel('Frame index')
    ax.set_ylabel('Image pixel std')
    ax.set_title('Effect of rolling window size on frame contrast stability')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"{'Window':<10} {'mean std':>10} {'std-of-std':>12}")
    print('-' * 34)
    for win, s in window_stds.items():
        print(f"{win:<10} {s.mean():>10.4f} {s.std():>12.4f}")

## 10. Cleanup (optional)

In [ ]:
import shutil

# Uncomment to delete everything under BASE_OUTPUT
# shutil.rmtree(BASE_OUTPUT, ignore_errors=True)
# print(f"Deleted {BASE_OUTPUT}")